# Exploring ConvNeXt with PyTorch

If we give a convolutional neural network a photograph of a rocket launch, what
happens between the pixels we feed in and the label we get back? In this exercise
we will see how the model labels one photograph and look at the size of the grid the network
uses at each stage. We will compare that run with a second photo, then run a
small block from ConvNeXt V2 to see what it adds to the architecture.

You can use a CPU runtime in Colab or a local Jupyter notebook with PyTorch and
its matching TorchVision package installed. The first run downloads a public
checkpoint of about 15 MB. You can run it without a Hugging Face account.

In [ ]:
%pip install --quiet --disable-pip-version-check timm==1.0.30 scikit-image==0.26.0 matplotlib==3.11.2

## Preparing the image and loading the model

For this example we will use ConvNeXt Atto, a compact image classifier with
about 3.7 million parameters. We load it with PyTorch Image Models, or
`timm`, a library that provides ready-to-use PyTorch vision models and their
image preprocessing settings.

Let us first load a launch photograph from scikit-image. Then we will use the
checkpoint's settings to resize the photo, crop it, and scale its color channels.
These steps put the image in the format the model expects.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
from skimage import data
import timm
import torch
from timm.data import ImageNetInfo

torch.set_num_threads(min(4, torch.get_num_threads()))

model_id = "timm/convnext_atto.d2_in1k"
revision = "435c0a8d02d5b872e90c8cde36887465982cb399"
model = timm.create_model(f"hf-hub:{model_id}@{revision}", pretrained=True)
model.eval()

photo = Image.fromarray(data.rocket())
data_config = timm.data.resolve_model_data_config(model)
preprocess = timm.data.create_transform(**data_config, is_training=False)
image_tensor = preprocess(photo).unsqueeze(0)

print("Input shape:", tuple(image_tensor.shape))

The dimensions are batch size, channels, height, and width. We have a batch of
one image with three color channels. Before we run the model, let us look at
the photo alongside the exact crop it will see. For display, we undo the scaling
of the color channels.

In [ ]:
output_dir = Path("outputs") / "convnext"
output_dir.mkdir(parents=True, exist_ok=True)

mean = torch.tensor(data_config["mean"])[:, None, None]
std = torch.tensor(data_config["std"])[:, None, None]
model_photo = (image_tensor[0] * std + mean).clamp(0, 1).permute(1, 2, 0)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.3), layout="constrained")
fig.suptitle("From photograph to model input", fontweight="bold")
axes[0].imshow(photo)
axes[0].set_title("Original photograph")
axes[1].imshow(model_photo)
axes[1].set_title(f"Model input ({model_photo.shape[0]} × {model_photo.shape[1]})")
for ax in axes:
    ax.set_axis_off()
fig.savefig(output_dir / "launch_photo_and_crop.png", dpi=300)
plt.show()

## Following an image through the network

We can now ask two questions about the same run. What labels does the model
give the highest probabilities, and how small is its grid by the time it makes that
decision? A feature map is an intermediate grid of numbers produced by the
network. At each grid location, the network keeps several learned features,
stored as channels. As we go deeper, will the number of channels shrink
along with the grid?

In [ ]:
with torch.inference_mode():
    last_features, stage_features = model.forward_intermediates(image_tensor)
    scores = model.forward_head(last_features)
    torch.testing.assert_close(scores, model(image_tensor))

for stage, feature_map in enumerate(stage_features, start=1):
    _, channels, height, width = feature_map.shape
    print(f"Stage {stage}: {height:>2} × {width:<2} grid, {channels:>3} channels")

labels = ImageNetInfo().label_descriptions()
probabilities, indices = scores[0].softmax(dim=0).topk(3)
print("\nTop 3 ImageNet labels")
for probability, index in zip(probabilities.tolist(), indices.tolist()):
    print(f"{probability:6.1%}  {labels[index]}")

Each stage uses a coarser grid and adds channels. The last grid still has
spatial structure. The classifier averages across the grid and uses the resulting
vector to score the 1,000 labels it was trained to predict. Those labels shape the
answer we get. In our run, the model ranked “drilling platform” above
“missile” and “space shuttle.” The photo shows a Falcon 9 launch, which has no
entry in the model's label list. The top score tells us which of the available
labels the model favored for this crop. A different photo gives us another way
to check how the labels and the grid sizes behave.

To see what these grids look like, let us display one channel from each stage.
We pick the channel whose values vary most across its grid so that we can see
its structure. Each panel has its own color scale. Compare the size of the
squares in the panels to see how much coarser the grid gets.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(8, 2.55), layout="constrained")
fig.suptitle("How the spatial grid changes across ConvNeXt stages",
             fontweight="bold")

for stage, (ax, features) in enumerate(zip(axes, stage_features), start=1):
    maps = features[0]
    channel = maps.flatten(1).std(dim=1, unbiased=False).argmax().item()
    _, height, width = maps.shape
    ax.imshow(maps[channel], cmap="viridis", interpolation="nearest")
    ax.set_title(f"Stage {stage}\n{height} × {width}, channel {channel}")
    ax.set_axis_off()

fig.supxlabel("One channel per stage. Dark to light shows low to high values "
             "within that panel.", fontsize=9)
fig.savefig(output_dir / "convnext_stage_grids.png", dpi=300)
plt.show()

## Comparing two photographs

Let us give the same model a second photograph, this time of a cat named
Chelsea. We will look at the crops that the model sees and compare one of the
feature maps we just plotted. We use Stage 2, channel 75 for both photographs.
That was the channel shown for the rocket in Stage 2 above. Using the same
channel and the same color scale lets us see how its values change when we
give the model a different image.

In [ ]:
photos = (data.rocket(), data.chelsea())
comparison_inputs = torch.stack(
    [preprocess(Image.fromarray(pixels)) for pixels in photos]
)

with torch.inference_mode():
    final_features, comparison_stages = model.forward_intermediates(comparison_inputs)
    cat_scores = model.forward_head(final_features)[1]

stage, channel = 2, 75
maps = comparison_stages[stage - 1][:, channel]
height, width = maps.shape[-2:]
low, high = maps.min().item(), maps.max().item()
crops = (comparison_inputs * std + mean).clamp(0, 1).permute(0, 2, 3, 1)

fig, axes = plt.subplots(1, 4, figsize=(8.4, 2.6), layout="constrained")
fig.suptitle("Two photographs through the same ConvNeXt channel",
             fontsize=16, fontweight="bold")
for name, crop, feature_map, photo_ax, map_ax in zip(
    ("Rocket", "Cat"), crops, maps, axes[::2], axes[1::2]
):
    photo_ax.imshow(crop)
    photo_ax.set_title(f"{name} crop", fontsize=13)
    heatmap = map_ax.imshow(
        feature_map, cmap="viridis", interpolation="nearest", vmin=low, vmax=high
    )
    map_ax.set_title(f"{name} feature map", fontsize=13)
    photo_ax.set_axis_off()
    map_ax.set_axis_off()

scale = fig.colorbar(heatmap, ax=axes, fraction=0.028, shrink=0.84, pad=0.025)
scale.set_label("Feature value", fontsize=12)
scale.ax.tick_params(labelsize=11)
fig.supxlabel(f"Model crops: 224 × 224. Feature maps: Stage {stage}, "
             f"channel {channel}, {height} × {width}.", fontsize=12)
fig.savefig(output_dir / "rocket_and_cat_comparison.png", dpi=300)
plt.show()

cat_probabilities, cat_indices = cat_scores.softmax(dim=0).topk(3)
print("Top 3 ImageNet labels for Chelsea")
for probability, index in zip(cat_probabilities.tolist(), cat_indices.tolist()):
    print(f"{probability:6.1%}  {labels[index]}")

The feature map has 28 rows and 28 columns in both cases. The architecture
determines its size. The patterns and the class scores change because the two
photographs give the network different values to work with.

## Trying a ConvNeXt V2 block

So far we have followed the whole image through a pretrained ConvNeXt model.
The V2 design keeps a similar sequence of stages and adds an operation called
*global response normalization* (GRN) inside its blocks. GRN measures how strongly
each channel responds across the whole grid, then uses those measurements to
adjust the channels relative to one another.

Let us take one V2 block and pass a small tensor through it. We will use random
weights and random input to check the block's shape and gradients. This check
illustrates how the block fits in a trainable network.

In [ ]:
torch.manual_seed(7)
v2 = timm.create_model("convnextv2_atto", pretrained=False)
block = v2.stages[0].blocks[0]
block_input = torch.randn(1, v2.feature_info[0]["num_chs"], 14, 14,
                          requires_grad=True)
block_output = block(block_input)
block_output.square().mean().backward()

grn = block.mlp.grn
assert block_output.shape == block_input.shape
assert block_input.grad is not None and torch.isfinite(block_input.grad).all()
assert grn.weight.grad is not None
assert torch.isfinite(grn.weight.grad).all()
assert grn.weight.grad.abs().sum().item() > 0

print("Input shape: ", tuple(block_input.shape))
print("Output shape:", tuple(block_output.shape))
print("Normalization layer in the block:", type(grn).__name__)
print("GRN scale gradient norm:", f"{grn.weight.grad.norm().item():.6g}")

The block keeps the grid size and channel count, so it can add its input back
to the transformed result, just as in the residual connection we studied
earlier. We also found a gradient on the GRN scale parameter. An optimizer
could use that gradient in a training run.

## Sources

1. Liu et al., [*A ConvNet for the 2020s*](https://arxiv.org/abs/2201.03545).
2. [ConvNeXt Atto checkpoint and license](https://huggingface.co/timm/convnext_atto.d2_in1k).
3. [SpaceX launch photograph via scikit-image](https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.rocket).
4. Woo et al., [*ConvNeXt V2*](https://arxiv.org/abs/2301.00808).
5. [Chelsea photograph by Stefan van der Walt](https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.chelsea).